In [53]:
from __future__ import annotations
import polars as pl
import polars.selectors as cs
import duckdb
from pathlib import Path
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output, callback, State, Patch, ctx, no_update
import dash
from datetime import date, timedelta

In [30]:
con = duckdb.connect("trips.duckdb")
con.execute("SET memory_limit = '10GB'")

path = Path.cwd()
if not Path(path, "data").exists(): path = path.parent
path_yellow = path / "data" / "yellow_taxi_unified.parquet"
path_green = path / "data" / "green_taxi_unified.parquet"
path_uber = path / "data" / "uber_unified.parquet"
path_agg_data = path / "data" / "data_reports_monthly.csv"
path_zone_lookup = path / "data" / "taxi_zone_lookup.csv"
path_climate = path / "data" / "NYC_Central_Park_weather_1869-2022.csv"

In [ ]:
color_map = {
    "Yellow": "#F2C200",
    "Green":  "#2E8B57",
    "FHV - High Volume": "#7B68EE",
    "FHV": "#7B68EE",
    "Yellow Taxi": "#F2C200", "Green Taxi":  "#2E8B57", "HVFHS": "#7B68EE"
}

# Gráficos Sobre los datos Agregados

In [4]:
res1 = con.execute(f"""
    SELECT
    CAST(strptime("Month/Year" || '-01', '%Y-%m-%d') AS DATE) AS month_year_date,
    "Avg Hours Per Day Per Vehicle" as m_hours,
    "License Class" as type
    FROM '{path_agg_data}'
    WHERE "License Class" IN ('Yellow', 'Green', 'FHV - High Volume')    
    ORDER BY month_year_date DESC;

""").pl()

fig = px.line(
    res1,
    x="month_year_date",
    y="m_hours",
    color="type",
    title="Active hours per day per vehicle",
    color_discrete_map=color_map,
)

fig.update_xaxes(dtick="M12", tickformat="%Y")

fig.update_traces(
    hovertemplate="Date: %{x|%b %Y}<br>Mean Hours: %{y}<extra>%{fullData.name}</extra>"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Mean Hours",
)

fig.show()

La gráfica muestra una tendencia bajista en el número de horas registradas por día por cada vehículo(licencia). Entre los años 2010-2014 abundaba el "doble-turno" en el que dos taxistas se ponían de acuerdo para compartir una misma licencia. Tendencia que ha ido en decremento dada la dificultad para coordinar los turnos y realizar el intercambio en una ciudad con tanto movimiento como Nueva York.

In [5]:
# Trip-in-progress hours per day per vehicle
res2 = con.execute(f"""
    SELECT
    CAST(strptime("Month/Year" || '-01', '%Y-%m-%d') AS DATE) AS month_year_date,
    try_cast(replace("Unique Vehicles", ',', '') AS BIGINT) AS unique_vehicles,
    "License Class" as type
    FROM '{path_agg_data}'
    WHERE "License Class" IN ('Yellow', 'Green', 'FHV - High Volume')
    ORDER BY month_year_date DESC;

""").pl()

fig = px.line(
    res2,
    x="month_year_date",
    y="unique_vehicles",
    color="type",
    title="Unique Vehicles Registered",
    color_discrete_map=color_map,
)

fig.update_xaxes(dtick="M12", tickformat="%Y")

fig.update_traces(
    hovertemplate="Date: %{x|%b %Y}<br>Unique Vehicles: %{y}<extra>%{fullData.name}</extra>"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Unique Vehicles",
)

fig.show()

Se aprecia una caida en le número de vehículos en el año 2020 por la pandemia. En general, el número de taxis amarillos y FHV está aumentando aunque se ve que ambos chocan con un techo que se puede corresponder con los niveles pre-pandémicos. No caben más coches en Nueva York.

In [79]:
res3 = con.execute(f"""
WITH
daily_fhv AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(EXTRACT(EPOCH FROM (dropoff_datetime - pickup_datetime))) / 60.0 AS daily_avg
  FROM '{path_uber}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_fhv AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'FHV - High Volume' AS type
  FROM daily_fhv
  GROUP BY 1
),


daily_yellow AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(EXTRACT(EPOCH FROM (dropoff_datetime - pickup_datetime))) / 60.0 AS daily_avg
  FROM '{path_yellow}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_yellow AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'Yellow' AS type
  FROM daily_yellow
  GROUP BY 1
),


daily_green AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(EXTRACT(EPOCH FROM (dropoff_datetime - pickup_datetime))) / 60.0 AS daily_avg
  FROM '{path_green}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_green AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'Green' AS type
  FROM daily_green
  GROUP BY 1
)


SELECT * FROM monthly_fhv
WHERE month >= DATE '2021-01-01' AND month < DATE '2026-01-01'
UNION ALL
SELECT * FROM monthly_yellow
WHERE month >= DATE '2021-01-01' AND month < DATE '2025-12-01'
UNION ALL
SELECT * FROM monthly_green
WHERE month >= DATE '2021-01-01' AND month < DATE '2025-12-01'
ORDER BY month, type;
""").pl()

fig = px.line(
    res3,
    x="month",
    y="monthly_avg",
    color="type",
    title="Trip duration (avg) evolution",
    color_discrete_map=color_map,
)


fig.update_xaxes(dtick="M12", tickformat="%Y")
fig.update_layout(
    yaxis_tickformat=",",
)


fig.update_traces(
    hovertemplate="Month: %{x|%b %Y}<br>Trip duration (avg): %{y:.1f} min<extra>%{fullData.name}</extra>"
)


fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Minutes",
)


fig.show()

Vemos que las distribuciones son prácticamente uniformes a lo largo de los años y muy parecidas. En 2021 se aprecia que la duración media de los viajes de taxis verdes era superior, lo que se puede explicar con que el primer año posterior al covid, la gente se movía entre distritos en taxi verde en vez de usar transporte público.
También hay una caida puntual sobre los taxis amarillos en octubre de 2023. Se considerará un error en los datos o un evento desconocido.

In [6]:
res3 = con.execute(f"""
WITH
daily_fhv AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(total_amount) AS daily_avg
  FROM '{path_uber}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_fhv AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'FHV - High Volume' AS type
  FROM daily_fhv
  GROUP BY 1
),

daily_yellow AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(total_amount) AS daily_avg
  FROM '{path_yellow}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_yellow AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'Yellow' AS type
  FROM daily_yellow
  GROUP BY 1
),

daily_green AS (
  SELECT
    CAST(pickup_datetime AS DATE) AS day,
    AVG(total_amount) AS daily_avg
  FROM '{path_green}'
  WHERE CAST(pickup_datetime AS DATE) >= DATE '2021-01-01'
    AND CAST(pickup_datetime AS DATE) <  DATE '2026-01-01'
  GROUP BY 1
),
monthly_green AS (
  SELECT
    CAST(date_trunc('month', day) AS DATE) AS month,
    AVG(daily_avg) AS monthly_avg,
    'Green' AS type
  FROM daily_green
  GROUP BY 1
)

SELECT * FROM monthly_fhv
WHERE month >= DATE '2021-01-01' AND month < DATE '2026-01-01'
UNION ALL
SELECT * FROM monthly_yellow
WHERE month >= DATE '2021-01-01' AND month < DATE '2025-12-01'
UNION ALL
SELECT * FROM monthly_green
WHERE month >= DATE '2021-01-01' AND month < DATE '2025-12-01'
ORDER BY month, type;
""").pl()


fig = px.line(
    res3,
    x="month",
    y="monthly_avg",
    color="type",
    title="Precio por viaje medio diario (No incluye propinas)",
    color_discrete_map=color_map,
)

fig.update_xaxes(dtick="M12", tickformat="%Y")
fig.update_layout(
    yaxis_tickprefix="$",
    yaxis_tickformat=",",   # 1,234.56 (cambia a ",.0f" si no quieres decimales)
)

fig.update_traces(
    hovertemplate="Date: %{x|%b %Y}<br>Paga media diaria: $%{y:,.2f}<extra>%{fullData.name}</extra>"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Paga media diaria",
)

fig.show()

Esta gráfica recoge datos desde 2021 y, suponiendo que la distribución de la duración de los viajes media se mantiene constante a lo largo de los años (gráfica anterior), vemos una tendencia alcista de los precios como es normal por la inflación. Por otra parte, vemos que los FHV suelen ser más caros en NY.

# Gráficos Dash: explorar demanda por zona y fecha

In [14]:
caminos = list(zip([path_yellow, path_green, path_uber], ["Yellow Taxi", "Green Taxi", "HVFHS"]))
cache = {}

for camino, tipo in caminos:
    for grain in ["month", "day"]:
        cache[(tipo, grain)] = con.execute(f"""
        WITH trips_enriched AS (
        SELECT
            date_trunc('{grain}', y.pickup_datetime) AS bucket,
            COALESCE(z.Zone, 'Unknown') AS zone
        FROM '{camino}' y
        LEFT JOIN '{path_zone_lookup}' z
            ON y.PULocationID = z.LocationID
        WHERE y.pickup_datetime >= DATE '2021-01-01'
            AND y.pickup_datetime <  DATE '2025-12-01'
        )
        SELECT
        bucket,
        zone,
        COUNT(*) AS trips,
        ? as type
        FROM trips_enriched
        GROUP BY bucket, zone, type
        ORDER BY bucket ASC;
        """, [tipo]).pl()

In [21]:
def _hacer_consulta(camino, tipo, zone, start_date, end_date, grain):
    key = (tipo, grain)
    start_dt = date.fromisoformat(start_date)
    end_dt = date.fromisoformat(end_date)
    
    if key in cache:
        return cache[key].filter(
            (pl.col("bucket") >= start_dt) &
            (pl.col("bucket") < end_dt + pl.duration(days=1))
        ).filter(pl.col("zone") == zone)

    return con.execute(f"""
        WITH trips_enriched AS (
        SELECT
            date_trunc('{grain}', y.pickup_datetime) AS bucket,
            COALESCE(z.Zone, 'Unknown') AS zone
        FROM '{camino}' y
        LEFT JOIN '{path_zone_lookup}' z
            ON y.PULocationID = z.LocationID
        WHERE y.pickup_datetime >= CAST(? AS DATE)
            AND y.pickup_datetime <  CAST(? AS DATE) + INTERVAL 1 DAY
        )
        SELECT
        bucket,
        zone,
        COUNT(*) AS trips,
        ? as type
        FROM trips_enriched
        WHERE zone = ?
        GROUP BY bucket, zone, type
        ORDER BY bucket ASC;
        """, [start_date, end_date, tipo, zone]).pl()


def creat_zone_view(zone: str, start_date:str = '2021-01-01', end_date:str = '2025-12-01', con: DuckDBPyConnection = con):
    d0 = date.fromisoformat(start_date)
    d1 = date.fromisoformat(end_date)
    days = (d1 - d0).days


    # Decide dtick + formatos según rango
    DAY = 24 * 60 * 60 * 1000
    HOUR = 60 * 60 * 1000
    if days >= 365 * 5:
        dtick = "M12"
        x_tickformat = "%Y"
        x_hover = "%b %Y"
        grain = "month"
    elif days >= 365:
        dtick = "M1"
        x_tickformat = "%b\n%Y"
        x_hover = "%b %Y"
        grain = "month"
    elif days >= 60:
        dtick = 15 * DAY      # <- 15 días
        x_tickformat = "%d %b\n%Y"
        x_hover = "%Y-%m-%d"
        grain = "day"
    elif days >= 5:
        dtick = 1 * DAY       # <- 1 día
        x_tickformat = "%d %b\n%Y"
        x_hover = "%Y-%m-%d"
        grain = "day"
    else:
        dtick = 1 * HOUR      # <- 1 hora
        x_tickformat = "%H:%M\n%d %b"
        x_hover = "%Y-%m-%d %H:%M"
        grain = "hour"
    
    concatena = []
    for camino, tipo in caminos:
        concatena.append(_hacer_consulta(camino, tipo, zone, start_date, end_date, grain))

    consulta = pl.concat(concatena).to_pandas()

    fig = px.line(
        consulta,
        x="bucket",
        y="trips",
        color="type",
        title=f"Num Trips evolution in zone: {zone}",
        color_discrete_map=color_map,
    )

    fig.update_xaxes(dtick=dtick, tickformat=x_tickformat, ticklabelmode="period")    
    

    fig.update_traces(
        hovertemplate=(
            f"Date: %{{x|{x_hover}}}<br>"
            f"Total trips: %{{y}}"
            f"<extra>%{{fullData.name}}</extra>"
        )
    )

    fig.update_layout(
        xaxis_title="Date",
        yaxis_title="Total trips",
    )

    return fig

In [81]:
creat_zone_view("Battery Park City", '2021-10-01', '2025-10-02')

In [ ]:
def date_list(min_d, max_d):
    out = []
    d = min_d
    while d <= max_d:
        out.append(d.isoformat())
        d += timedelta(days=1)
    return out

zona_por_distrito = pl.read_csv(path_zone_lookup)
zona_por_distrito = (
    zona_por_distrito
    .group_by("Borough")
    .agg(
        pl.col("Zone").unique().alias("Zone")
    )
)
zona_por_distrito = {r["Borough"]: r["Zone"] for r in zona_por_distrito.to_dicts()}

ALL_DATES = date_list(date(2021,1,1), date(2025,12,31))
DATE_OPTIONS = [{"label": d, "value": d} for d in ALL_DATES]

# vista interactiva
app = Dash(__name__)
controls_row_style = {
    "display": "flex",
    "alignItems": "center",
    "gap": "10px",
    "flexWrap": "wrap",
    "padding": "6px",
    "backgroundColor": "#f8f9fa",
    "borderRadius": "10px",
}

card_style = {
    "border": "1px solid #ddd",
    "borderRadius": "12px",
    "backgroundColor": "white",
    "padding": "8px",
    "boxShadow": "0 4px 12px rgba(0,0,0,0.06)",
}


app.layout = html.Div([
    # Controles
    html.Div([
        html.Div([
            html.Div([
                html.Label("Municipio:", style={"font-weight": "bold"}),
                dcc.Dropdown(
                    options=list(zona_por_distrito.keys()),
                    id="muni-name",
                    value="Manhattan",
                    style={"width": "200px"},
                    clearable = False
                ),
            ]),

            html.Div([
                html.Label("Zona:", style={"font-weight": "bold"}),
                dcc.Dropdown(
                    options=zona_por_distrito["Manhattan"],
                    id="zona-name",
                    value=zona_por_distrito["Manhattan"][0],
                    style={"width": "200px"},
                    clearable = False
                ),
            ]),

            html.Div([
                html.Label("Inicio:", style={"font-weight": "bold"}),
                dcc.Dropdown(
                    id="start-date",
                    options=DATE_OPTIONS,
                    value="2021-01-01",
                    clearable=False,
                    style={"width": "160px"},
                ),
            ]),
            html.Div([
                html.Label("Fin:", style={"font-weight": "bold"}),
                dcc.Dropdown(
                    id="end-date",
                    options=DATE_OPTIONS,
                    value="2025-11-30",
                    clearable=False,
                    style={"width": "160px"},
                ),
            ]),
        ], style=controls_row_style),

        # Vista Zona
        html.Div([
            dcc.Graph(id="zone-fig", figure=creat_zone_view(zona_por_distrito["Manhattan"][0])),
        ], style={"marginTop": "12px"}),
    ], style={**card_style, "marginTop": "0px"}),

    
], style={"padding": "12px", "backgroundColor": "#f5f6f8"})


# Callback para actualizar el mapa
@callback(
    Output("zona-name", "options"),
    Output("zona-name", "value"),
    Output("zone-fig", "figure"),
    Input("muni-name", "value"),
    Input("zona-name", "value"),
    Input("start-date", "value"),
    Input("end-date", "value"),
)
def update_all(muni_dropdown, zona_value, start_date, end_date):
    muni = muni_dropdown
    trigger = ctx.triggered_id

    # 2) Normaliza y valida muni
    muni = muni = (muni or "Manhattan").strip()

    zona_opts = zona_por_distrito[muni]

    # 3) Zona válida
    if zona_value not in zona_opts:
        zona_value = zona_opts[0]

    # 4) Figura
    try:
        fig_zone = creat_zone_view(zona_value, start_date=start_date, end_date=end_date)
    except Exception:
        fig_zone = no_update

    return zona_opts, zona_value, fig_zone



if __name__ == '__main__':
    app.run(debug=True, port=8052)

# Gráficos con Dash: Cruze con datos meteorológicos

In [72]:
def _hacer_consulta_lluvia(camino, zone):
    return con.execute(f"""
        WITH trips_enriched AS (
        SELECT
            date_trunc('hour', y.pickup_datetime) as bucket,
            COALESCE(z.Zone, 'Unknown') AS zone,
            (c.PRCP > 0.41) AS lluvia,
        FROM '{camino}' y
        LEFT JOIN '{path_zone_lookup}' z
            ON y.PULocationID = z.LocationID
        LEFT JOIN '{path_climate}' c
            ON date_trunc('day', y.pickup_datetime) = CAST(c.DATE AS DATE)
        WHERE y.pickup_datetime >= DATE '2021-01-01'
            AND y.pickup_datetime <  DATE '2025-12-01'
        ),
        count_day AS (
            SELECT
            bucket,
            zone,
            lluvia,
            COUNT(*) AS trips
            FROM trips_enriched
            WHERE zone = ?
            GROUP BY bucket, zone, lluvia
            ORDER BY bucket ASC
        )
        SELECT 
            date_part('hour', bucket) AS hour,
            zone,
            AVG(trips) as avg_trips,
            lluvia,
            FROM count_day
            GROUP BY hour, zone, lluvia;
        """, [zone]).df()

def _hacer_consulta_nieve(camino, zone):
    return con.execute(f"""
        WITH trips_enriched AS (
        SELECT
            date_trunc('hour', y.pickup_datetime) as bucket,
            COALESCE(z.Zone, 'Unknown') AS zone,
            (c.SNOW > 0) AS nieve
        FROM '{camino}' y
        LEFT JOIN '{path_zone_lookup}' z
            ON y.PULocationID = z.LocationID
        LEFT JOIN '{path_climate}' c
            ON date_trunc('day', y.pickup_datetime) = CAST(c.DATE AS DATE)
        WHERE y.pickup_datetime >= DATE '2021-01-01'
            AND y.pickup_datetime <  DATE '2025-12-01'
        ),
        count_day AS (
            SELECT
            bucket,
            zone,
            nieve,
            COUNT(*) AS trips
            FROM trips_enriched
            WHERE zone = ?
            GROUP BY bucket, zone, nieve
            ORDER BY bucket ASC
        )
        SELECT 
            date_part('hour', bucket) AS hour,
            zone,
            AVG(trips) as avg_trips,
            nieve
            FROM count_day
            GROUP BY hour, zone, nieve;
        """, [zone]).df()

In [73]:
def plot_rain_snow(zone):
    df_lluvia = _hacer_consulta_lluvia(path_yellow, zone)
    pdf_lluvia = df_lluvia.dropna().sort_values("hour")
    pdf_lluvia["dash"] = ~ pdf_lluvia["lluvia"]
    
    df_nieve = _hacer_consulta_nieve(path_yellow, zone)
    pdf_nieve = df_nieve.dropna().sort_values("hour")
    pdf_nieve["dash"] = ~pdf_nieve["nieve"]
    

    RAIN_COLORS = {False: "#2E3440", True: "blue"}
    SNOW_COLORS = {False: "#2E3440", True: "red"}

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Rain', 'Snow'),
        shared_xaxes=True,
        vertical_spacing=0.08,
    )
    
    # SUBPLOT 1: lluvia
    fig_lluvia = px.line(
        pdf_lluvia, 
        x="hour", 
        y="avg_trips", 
        color="lluvia", 
        markers=True,
        color_discrete_map=RAIN_COLORS,
        line_dash="dash"
    )
    for trace in fig_lluvia.data:
        trace.showlegend = False
        fig.add_trace(trace, row=1, col=1)
    
    # SUBPLOT 2: nieve  
    fig_nieve = px.line(
        pdf_nieve, 
        x="hour", 
        y="avg_trips", 
        color="nieve", 
        color_discrete_map=SNOW_COLORS,
        markers=True,
        line_dash="dash"
    )
    for trace in fig_nieve.data:
        trace.showlegend = False
        fig.add_trace(trace, row=2, col=1)
    

    fig.update_xaxes(title="Hour of day", tickmode="linear", tick0=0, dtick=1, row=2, col=1)
    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray', row=1, col=1)
    
    fig.update_yaxes(title="Average trips", row=1, col=1)
    fig.update_yaxes(title="Average trips", row=2, col=1)
    
    fig.update_layout(height=700, title=f"Trips by hour - {zone}")
    return fig

g_rs = plot_rain_snow("Jamaica")
g_rs.show()

In [74]:
zona_por_distrito = pl.read_csv(path_zone_lookup)
zona_por_distrito = (
    zona_por_distrito
    .group_by("Borough")
    .agg(
        pl.col("Zone").unique().alias("Zone")
    )
)
zona_por_distrito = {r["Borough"]: r["Zone"] for r in zona_por_distrito.to_dicts()}

# vista interactiva
app = Dash(__name__)
controls_row_style = {
    "display": "flex",
    "alignItems": "center",
    "gap": "10px",
    "flexWrap": "wrap",
    "padding": "6px",
    "backgroundColor": "#f8f9fa",
    "borderRadius": "10px",
}

card_style = {
    "border": "1px solid #ddd",
    "borderRadius": "12px",
    "backgroundColor": "white",
    "padding": "8px",
    "boxShadow": "0 4px 12px rgba(0,0,0,0.06)",
}


app.layout = html.Div([
    # Controles
    html.Div([
        html.Div([
            html.Div([
                html.Label("Municipio:", style={"font-weight": "bold"}),
                dcc.Dropdown(
                    options=list(zona_por_distrito.keys()),
                    id="muni-name",
                    value="Manhattan",
                    style={"width": "200px"},
                    clearable = False
                ),
            ]),

            html.Div([
                html.Label("Zona:", style={"font-weight": "bold"}),
                dcc.Dropdown(
                    options=zona_por_distrito["Manhattan"],
                    id="zona-name",
                    value=zona_por_distrito["Manhattan"][0],
                    style={"width": "200px"},
                    clearable = False
                ),
            ]),
        ], style=controls_row_style),

        # Vista Zona
        html.Div([
            dcc.Graph(id="zone-fig", figure=plot_rain_snow(zona_por_distrito["Manhattan"][0])),
        ], style={"marginTop": "12px"}),
    ], style={**card_style, "marginTop": "0px"}),

    
], style={"padding": "12px", "backgroundColor": "#f5f6f8"})


# Callback para actualizar el mapa
@callback(
    Output("zona-name", "options"),
    Output("zona-name", "value"),
    Output("zone-fig", "figure"),
    Input("muni-name", "value"),
    Input("zona-name", "value"),
)
def update_all(muni_dropdown, zona_value):
    muni = muni_dropdown
    trigger = ctx.triggered_id

    # 2) Normaliza y valida muni
    muni = muni = (muni or "Manhattan").strip()

    zona_opts = zona_por_distrito[muni]

    # 3) Zona válida
    if zona_value not in zona_opts:
        zona_value = zona_opts[0]

    # 4) Figura
    try:
        fig_zone = plot_rain_snow(zona_value)
    except Exception:
        fig_zone = no_update

    return zona_opts, zona_value, fig_zone



if __name__ == '__main__':
    app.run(debug=True, port=8053)